# 11 — Quafu-SQC 一站式真机验证

这本 notebook 把 `09` 的 2-qubit IMDb flat-QOS 和 `10` 的 4-qubit
general-vector QSVT 真机流程合并到一个入口：

1. 重新审计 manifest、QASM hash、qubit/classical-bit/measurement schema；
2. 查询 Quafu-SQC 后端队列；
3. 按预注册顺序提交少量任务；
4. 保存 task receipt、raw JSON、corrected counts、QLisp 和 returned
   transpiled QASM；
5. 解析 flat $F_{\rm echo}=P(00)$；
6. 解析 full QSVT 的 heralding probability 与 conditional fidelity；
7. 统计远端编译后的 depth、二比特门数、二比特 depth 和 SWAP。

默认 `ACTION="dry_run"`，完整执行不会联网或提交任务。真正提交只需要修改下一
个配置单元；所有输出写入 `results/hardware_validation/`。

In [1]:
# =========================
# 用户只需要修改这个配置单元
# =========================

# 可选：dry_run | status | submit | submit_and_wait | fetch
ACTION = "dry_run"

BACKEND = "Baihua"
DEFAULT_COMPILER = "quarkcircuit"
USE_READOUT_CORRECTION = False
OPEN_DD = None                 # None | "XY4" | "CPMG"
TARGET_QUBITS = []             # 留空让编译器映射

# Quafu/Qiskit 通常返回 c[n-1]...c[0]。先跑 one-hot controls；若返回完全反向，
# 改成 "reversed" 后再解析。可选：qiskit | reversed
BITSTRING_ORDER = "qiskit"
REQUIRE_4Q_BIT_ORDER_CALIBRATION_FOR_QSVT = True

# 真正提交时必须改成精确字符串 RUN_QUAFU_HARDWARE。
CONFIRM_REAL_SUBMISSION = ""

# 防止误把整套 campaign 一次提交。
MAX_TASKS_PER_SUBMISSION_CELL = 2
ALLOW_DUPLICATE_SUBMISSION = False
ALLOW_DEEP_QSVT = False

# 轮询参数只在 submit_and_wait 时使用。
MAX_WAIT_SECONDS = 1800
POLL_INTERVAL_SECONDS = 10
RESULT_TIMEOUT_SECONDS = 60
FORCE_FETCH_NONFINISHED = False

# 默认只启用两条 2q one-hot 位序校准。每一阶段验收后，再开启下一项。
RUN_PLAN = [
    {
        "enabled": True,
        "run_label": "bitorder_2q_xq0",
        "experiment_id": "cal_bitorder_2q_x_q0",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": True,
        "run_label": "bitorder_2q_xq1",
        "experiment_id": "cal_bitorder_2q_x_q1",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "bitorder_4q_xq0",
        "experiment_id": "cal_bitorder_4q_x_q0",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "bitorder_4q_xq3",
        "experiment_id": "cal_bitorder_4q_x_q3",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "smoke_h_only_qc",
        "experiment_id": "imdb_flat_001_h_only_M0_r0",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "control_synthetic_entangled",
        "experiment_id":
            "imdb_flat_005_synthetic_entangled_balanced_M8_r0",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "control_exact_pair_qc",
        "experiment_id": "imdb_flat_002_exact_pair_M0_r0",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "control_balanced_qc",
        "experiment_id": "imdb_flat_003_balanced_stream_M16_r0",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "control_balanced_qsteed",
        "experiment_id": "imdb_flat_003_balanced_stream_M16_r0",
        "compiler": "qsteed",
    },
    {
        "enabled": False,
        "run_label": "control_phase_probe",
        "experiment_id": "imdb_flat_004_phase_count_probe_M16_r0",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "imdb_flat_iid_m4_r1",
        "experiment_id": "imdb_flat_007_iid_M4_r1",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "imdb_flat_iid_m16_r7",
        "experiment_id": "imdb_flat_033_iid_M16_r7",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "imdb_flat_iid_m32_r10",
        "experiment_id": "imdb_flat_056_iid_M32_r10",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "general_qsvt_m8_prepare",
        "experiment_id": "imdb_d4_general_qsvt_m8_prepare",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "general_qsvt_m8_verify",
        "experiment_id": "imdb_d4_general_qsvt_m8_verify",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "general_qsvt_m16_prepare",
        "experiment_id": "imdb_d4_general_qsvt_m16_prepare",
        "compiler": "quarkcircuit",
    },
    {
        "enabled": False,
        "run_label": "general_qsvt_m16_verify",
        "experiment_id": "imdb_d4_general_qsvt_m16_verify",
        "compiler": "quarkcircuit",
    },
]

# 重新打开 notebook 后取回结果：填写已提交的 run_label。
# 用 ["*"] 取回 registry 中所有任务。
FETCH_RUN_LABELS = []

# registry 不可用时可手工填写；task_id 必须是正整数。
MANUAL_FETCH_PLAN = [
    # {
    #     "run_label": "smoke_h_only_qc",
    #     "experiment_id": "imdb_flat_001_h_only_M0_r0",
    #     "compiler": "quarkcircuit",
    #     "task_id": 123456789,
    # },
]

## 1. 环境与路径

notebook 会自动判断当前工作目录是项目根目录还是 `notebooks/`，因此从
JupyterLab 打开或用 `nbconvert` 执行都可以。真机接口使用当前环境中已验证的
`quafusqc` 发行包，其导入路径为 `from quafu import Task`。

In [2]:
import hashlib
import json
import math
import os
import platform
import re
import time
from datetime import datetime, timezone
from getpass import getpass
from importlib.metadata import version
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", "/private/tmp/qos-hardware-mplconfig"
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from qiskit import qasm2
from qiskit.circuit import Gate


def resolve_project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (
            (candidate / "results/imdb_vector_pilot/manifest.json").exists()
            and (
                candidate
                / "results/general_vector_qsvt_pilot/manifest.json"
            ).exists()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        "找不到 Quafu 项目根目录。请从项目根目录或 notebooks/ 启动。"
    )


PROJECT_ROOT = resolve_project_root()
FLAT_MANIFEST_PATH = (
    PROJECT_ROOT / "results/imdb_vector_pilot/manifest.json"
)
GENERAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / "results/general_vector_qsvt_pilot/manifest.json"
)
RESULT_ROOT = PROJECT_ROOT / "results/hardware_validation"
RAW_RESULT_DIR = RESULT_ROOT / "raw_results"
CALIBRATION_QASM_DIR = RESULT_ROOT / "calibration_qasm"
REGISTRY_PATH = RESULT_ROOT / "task_registry.json"
HARDWARE_CSV_PATH = RESULT_ROOT / "hardware_results.csv"
HARDWARE_JSON_PATH = RESULT_ROOT / "hardware_results.json"

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
CALIBRATION_QASM_DIR.mkdir(parents=True, exist_ok=True)

ENVIRONMENT = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "qiskit": version("qiskit"),
    "quafusqc": version("quafusqc"),
}

print("Project root:", PROJECT_ROOT)
print("Action:", ACTION)
display(pd.Series(ENVIRONMENT, name="version").to_frame())

Project root: /Users/tangzhi/.codex/.chatgpt-projects/g-p-6a5f771ad30c8191a3fc5b9cc6ea6d75/Quafu_stage
Action: dry_run


,version
python,3.13.7
numpy,2.3.3
pandas,2.3.2
qiskit,2.5.1
quafusqc,3.3.9


## 2. 离线审计全部候选线路

这里不相信旧的 manifest 结论，而是重新读取每一条 QASM 并检查：

- 文件 SHA-256；
- OpenQASM 2.0 能否 round-trip parse；
- 现场生成的 2q/4q one-hot bit-order controls；
- flat 为 2q/2c/2measure，full QSVT 为 4q/4c/4measure；
- `measure q[i] -> c[i]` 的 classical-bit 顺序；
- shots 是 1024 的整数倍；
- full QSVT target 文件 hash。

后续 counts parser 依赖 Qiskit/Quafu 显示 bitstring 为高位 classical bit 在左：
4-bit full QSVT 中字符串为 `c3 c2 c1 c0`，所以前两位 `c3c2=00` 表示
`real=0, signal=0` 的 heralding 成功分支。

In [3]:
def sha256_bytes(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    raise TypeError(f"Cannot JSON-encode {type(value).__name__}")


def write_json_atomic(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            default=json_default,
        ),
        encoding="utf-8",
    )
    temporary.replace(path)


def path_for_record(path):
    path = Path(path).resolve()
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def build_bit_order_qasm(n_qubits, x_qubit):
    return (
        "OPENQASM 2.0;\n"
        'include "qelib1.inc";\n'
        f"qreg q[{n_qubits}];\n"
        f"creg c[{n_qubits}];\n"
        f"x q[{x_qubit}];\n"
        "measure q -> c;\n"
    )


calibration_entries = []
for n_qubits, x_qubit in [(2, 0), (2, 1), (4, 0), (4, 3)]:
    experiment_id = (
        f"cal_bitorder_{n_qubits}q_x_q{x_qubit}"
    )
    qasm_path = (
        CALIBRATION_QASM_DIR / f"{experiment_id}.qasm"
    )
    qasm_text = build_bit_order_qasm(n_qubits, x_qubit)
    qasm_path.write_text(qasm_text, encoding="utf-8")
    calibration_entries.append(
        {
            "experiment_id": experiment_id,
            "control_type": "bit_order_one_hot",
            "n_total_qubits": n_qubits,
            "x_qubit": x_qubit,
            "ideal_expected_bitstring": format(
                1 << x_qubit, f"0{n_qubits}b"
            ),
            "ideal_expected_probability": 1.0,
            "S_shots": 1024,
            "logical_depth": 2,
            "logical_cx": 0,
            "qasm_path": str(qasm_path.relative_to(PROJECT_ROOT)),
            "qasm_sha256": hashlib.sha256(
                qasm_text.encode("utf-8")
            ).hexdigest(),
        }
    )


flat_manifest = json.loads(FLAT_MANIFEST_PATH.read_text())
general_manifest = json.loads(GENERAL_MANIFEST_PATH.read_text())

all_entries = []
for item in calibration_entries:
    all_entries.append(
        {**item, "_family": "bit_order_calibration"}
    )
for item in flat_manifest:
    all_entries.append({**item, "_family": "flat"})
for item in general_manifest:
    all_entries.append({**item, "_family": "general_qsvt"})

entry_by_id = {}
for entry in all_entries:
    experiment_id = entry["experiment_id"]
    if experiment_id in entry_by_id:
        raise AssertionError(f"Duplicate experiment id: {experiment_id}")
    entry_by_id[experiment_id] = entry

audit_rows = []
for entry in all_entries:
    qasm_path = PROJECT_ROOT / entry["qasm_path"]
    expected_qubits = int(
        entry.get(
            "n_total_qubits",
            entry.get("logical_qubits"),
        )
    )
    observed_hash = sha256_bytes(qasm_path)
    hash_ok = observed_hash == entry["qasm_sha256"]
    parse_error = ""
    measurement_map = []
    try:
        circuit = qasm2.loads(qasm_path.read_text(encoding="utf-8"))
        for instruction in circuit.data:
            if instruction.operation.name == "measure":
                q_index = circuit.find_bit(
                    instruction.qubits[0]
                ).index
                c_index = circuit.find_bit(
                    instruction.clbits[0]
                ).index
                measurement_map.append((q_index, c_index))
        measurement_map = sorted(measurement_map)
        schema_ok = (
            circuit.num_qubits == expected_qubits
            and circuit.num_clbits == expected_qubits
            and len(measurement_map) == expected_qubits
        )
        measurement_order_ok = measurement_map == [
            (index, index) for index in range(expected_qubits)
        ]
    except Exception as exc:
        circuit = None
        schema_ok = False
        measurement_order_ok = False
        parse_error = f"{type(exc).__name__}: {exc}"

    target_hash_ok = True
    if entry["_family"] == "general_qsvt":
        target_path = PROJECT_ROOT / entry["target_path"]
        target_hash_ok = (
            sha256_bytes(target_path) == entry["target_sha256"]
        )

    shots = int(entry["S_shots"])
    shots_ok = shots > 0 and shots % 1024 == 0
    passed = all(
        [
            qasm_path.exists(),
            hash_ok,
            schema_ok,
            measurement_order_ok,
            target_hash_ok,
            shots_ok,
        ]
    )
    audit_rows.append(
        {
            "experiment_id": entry["experiment_id"],
            "family": entry["_family"],
            "qasm_path": entry["qasm_path"],
            "expected_qubits": expected_qubits,
            "qasm_sha256": observed_hash,
            "hash_ok": hash_ok,
            "schema_ok": schema_ok,
            "measurement_order_ok": measurement_order_ok,
            "measurement_map": measurement_map,
            "target_hash_ok": target_hash_ok,
            "shots": shots,
            "shots_ok": shots_ok,
            "parse_error": parse_error,
            "audit_pass": passed,
        }
    )

audit_df = pd.DataFrame(audit_rows)
if not bool(audit_df["audit_pass"].all()):
    display(audit_df.loc[~audit_df["audit_pass"]])
    raise AssertionError("至少一条 QASM 未通过离线审计。")

offline_audit_payload = {
    "audited_at_utc": datetime.now(timezone.utc).isoformat(),
    "network_used": False,
    "hardware_submitted": False,
    "environment": ENVIRONMENT,
    "flat_manifest_sha256": sha256_bytes(FLAT_MANIFEST_PATH),
    "general_manifest_sha256": sha256_bytes(
        GENERAL_MANIFEST_PATH
    ),
    "total_qasm": int(len(audit_df)),
    "bit_order_calibration_qasm": int(
        (audit_df["family"] == "bit_order_calibration").sum()
    ),
    "flat_qasm": int((audit_df["family"] == "flat").sum()),
    "general_qsvt_qasm": int(
        (audit_df["family"] == "general_qsvt").sum()
    ),
    "failed_qasm": int((~audit_df["audit_pass"]).sum()),
}
write_json_atomic(
    RESULT_ROOT / "offline_audit.json",
    offline_audit_payload,
)
audit_df.to_csv(RESULT_ROOT / "offline_audit.csv", index=False)

print(
    "Offline QASM audit PASS:",
    len(audit_df),
    "circuits; failures:",
    int((~audit_df["audit_pass"]).sum()),
)

Offline QASM audit PASS: 73 circuits; failures: 0


## 3. 冻结本轮运行计划

每个 `run_label` 必须唯一。相同线路可以分别用 `quarkcircuit` 与 `qsteed`
提交，但要使用不同 label。默认最多一次提交两条任务；full QSVT 还需要额外设置
`ALLOW_DEEP_QSVT=True`。

In [4]:
ALLOWED_ACTIONS = {
    "dry_run", "status", "submit", "submit_and_wait", "fetch"
}
ALLOWED_COMPILERS = {
    None, "quarkcircuit", "qsteed", "qiskit"
}
if ACTION not in ALLOWED_ACTIONS:
    raise ValueError(f"Unknown ACTION: {ACTION!r}")
if BITSTRING_ORDER not in {"qiskit", "reversed"}:
    raise ValueError(
        "BITSTRING_ORDER must be 'qiskit' or 'reversed'."
    )
if OPEN_DD not in {None, "XY4", "CPMG"}:
    raise ValueError("OPEN_DD must be None, 'XY4', or 'CPMG'.")
if (
    any(
        not isinstance(item, (int, np.integer))
        or isinstance(item, (bool, np.bool_))
        or int(item) < 0
        for item in TARGET_QUBITS
    )
    or len(TARGET_QUBITS) != len(set(TARGET_QUBITS))
):
    raise ValueError(
        "TARGET_QUBITS must contain unique nonnegative integers."
    )

labels = [item["run_label"] for item in RUN_PLAN]
if len(labels) != len(set(labels)):
    raise ValueError("RUN_PLAN contains duplicate run_label values.")

plan_rows = []
for item in RUN_PLAN:
    label = item["run_label"]
    if not re.fullmatch(r"[A-Za-z0-9_-]+", label):
        raise ValueError(f"Unsafe run_label: {label!r}")
    experiment_id = item["experiment_id"]
    if experiment_id not in entry_by_id:
        raise ValueError(
            f"Experiment id is not in either manifest: {experiment_id}"
        )
    compiler = item.get("compiler", DEFAULT_COMPILER)
    if compiler not in ALLOWED_COMPILERS:
        raise ValueError(f"Unsupported compiler: {compiler!r}")
    entry = entry_by_id[experiment_id]
    audit = audit_df.loc[
        audit_df["experiment_id"] == experiment_id
    ].iloc[0]
    plan_rows.append(
        {
            "enabled": bool(item["enabled"]),
            "run_label": label,
            "experiment_id": experiment_id,
            "family": entry["_family"],
            "control_type": entry.get(
                "control_type", entry.get("circuit_kind")
            ),
            "compiler": compiler,
            "shots": int(entry["S_shots"]),
            "ideal_F_echo": entry.get("ideal_F_echo"),
            "ideal_expected_bitstring": entry.get(
                "ideal_expected_bitstring"
            ),
            "ideal_expected_probability": entry.get(
                "ideal_expected_probability"
            ),
            "ideal_success_probability": entry.get(
                "ideal_success_probability"
            ),
            "ideal_conditional_fidelity": entry.get(
                "ideal_conditional_fidelity"
            ),
            "local_compiled_depth": entry.get(
                "compiled_depth", entry.get("logical_depth")
            ),
            "local_two_qubit_gates": entry.get(
                "compiled_two_qubit_gates",
                entry.get("logical_cx"),
            ),
            "qasm_path": entry["qasm_path"],
            "qasm_sha256": audit["qasm_sha256"],
            "risk": (
                "deep"
                if entry["_family"] == "general_qsvt"
                else "low_or_medium"
            ),
        }
    )

plan_df = pd.DataFrame(plan_rows)
plan_df.to_csv(RESULT_ROOT / "audited_run_plan.csv", index=False)
display(
    plan_df[
        [
            "enabled",
            "run_label",
            "experiment_id",
            "compiler",
            "shots",
            "ideal_expected_bitstring",
            "ideal_F_echo",
            "ideal_success_probability",
            "ideal_conditional_fidelity",
            "local_compiled_depth",
            "local_two_qubit_gates",
            "risk",
        ]
    ]
)
print(
    "Enabled tasks:",
    int(plan_df["enabled"].sum()),
    "| ACTION:",
    ACTION,
)

,enabled,run_label,experiment_id,compiler,shots,ideal_expected_bitstring,ideal_F_echo,ideal_success_probability,ideal_conditional_fidelity,local_compiled_depth,local_two_qubit_gates,risk
0,True,bitorder_2q_xq0,cal_bitorder_2q_x_q0,quarkcircuit,1024,01,NaN,NaN,NaN,2,0,low_or_medium
1,True,bitorder_2q_xq1,cal_bitorder_2q_x_q1,quarkcircuit,1024,10,NaN,NaN,NaN,2,0,low_or_medium
2,False,bitorder_4q_xq0,cal_bitorder_4q_x_q0,quarkcircuit,1024,0001,NaN,NaN,NaN,2,0,low_or_medium
3,False,bitorder_4q_xq3,cal_bitorder_4q_x_q3,quarkcircuit,1024,1000,NaN,NaN,NaN,2,0,low_or_medium
4,False,smoke_h_only_qc,imdb_flat_001_h_only_M0_r0,quarkcircuit,2048,None,1.000000,NaN,NaN,3,0,low_or_medium
5,False,control_synthetic_entangled,imdb_flat_005_synthetic_entangled_balanced_M8_r0,quarkcircuit,2048,None,1.000000,NaN,NaN,18,6,low_or_medium
6,False,control_exact_pair_qc,imdb_flat_002_exact_pair_M0_r0,quarkcircuit,2048,None,1.000000,NaN,NaN,37,12,low_or_medium
7,False,control_balanced_qc,imdb_flat_003_balanced_stream_M16_r0,quarkcircuit,2048,None,1.000000,NaN,NaN,88,30,low_or_medium
8,False,control_balanced_qsteed,imdb_flat_003_balanced_stream_M16_r0,qsteed,2048,None,1.000000,NaN,NaN,88,30,low_or_medium
9,False,control_phase_probe,imdb_flat_004_phase_count_probe_M16_r0,quarkcircuit,2048,None,0.728553,NaN,NaN,87,30,low_or_medium


Enabled tasks: 2 | ACTION: dry_run


## 4. 统一结果解析器与自检

raw `count` 必须是非负整数 counts；`corrected` 允许非负浮点权重。
returned QASM 的 barrier 和 measurement 不会被误计成二比特门。

对于 full verifier：

$$
p_{\rm succ}=P(c_3c_2=00),
\qquad
F_{\rm cond}=\frac{P(0000)}{P(c_3c_2=00)}.
$$

`prepare` 线路只能测 $p_{\rm succ}$；没有 target inverse，不能从全零地址概率
冒充 conditional fidelity。

In [5]:
def empty_distribution(reason):
    return {
        "weights": {},
        "returned_weights": {},
        "total_weight": 0.0,
        "dominant_returned_bitstring": None,
        "dominant_canonical_bitstring": None,
        "parse_reason": reason,
    }


def normalize_distribution(
    values,
    n_bits,
    *,
    require_integer,
):
    if BITSTRING_ORDER not in {"qiskit", "reversed"}:
        return empty_distribution(
            f"invalid_BITSTRING_ORDER:{BITSTRING_ORDER!r}"
        )
    if not isinstance(values, dict) or not values:
        return empty_distribution("missing_counts")
    cleaned = {}
    returned_cleaned = {}
    for raw_bits, raw_value in values.items():
        returned_bits = str(raw_bits).replace(" ", "")
        if (
            len(returned_bits) != n_bits
            or set(returned_bits) - {"0", "1"}
        ):
            return empty_distribution(
                f"invalid_bitstring:{raw_bits}"
            )
        if isinstance(raw_value, (bool, np.bool_)):
            return empty_distribution(
                f"invalid_weight:{raw_bits}:{raw_value!r}"
            )
        try:
            value = float(raw_value)
        except (TypeError, ValueError, OverflowError):
            return empty_distribution(
                f"invalid_weight:{raw_bits}:{raw_value!r}"
            )
        if not math.isfinite(value) or value < 0:
            return empty_distribution(
                f"invalid_weight:{raw_bits}:{raw_value!r}"
            )
        if require_integer and not value.is_integer():
            return empty_distribution(
                f"noninteger_raw_count:{raw_bits}:{raw_value!r}"
            )
        bits = (
            returned_bits
            if BITSTRING_ORDER == "qiskit"
            else returned_bits[::-1]
        )
        cleaned[bits] = cleaned.get(bits, 0.0) + value
        returned_cleaned[returned_bits] = (
            returned_cleaned.get(returned_bits, 0.0) + value
        )
    total = float(sum(cleaned.values()))
    if total <= 0:
        return empty_distribution("zero_total_counts")
    dominant_returned = max(
        returned_cleaned, key=returned_cleaned.get
    )
    dominant_canonical = (
        dominant_returned
        if BITSTRING_ORDER == "qiskit"
        else dominant_returned[::-1]
    )
    return {
        "weights": cleaned,
        "returned_weights": returned_cleaned,
        "total_weight": total,
        "dominant_returned_bitstring": dominant_returned,
        "dominant_canonical_bitstring": dominant_canonical,
        "parse_reason": "ok",
    }


def probability(distribution, predicate):
    if distribution["parse_reason"] != "ok":
        return np.nan
    return sum(
        value
        for bits, value in distribution["weights"].items()
        if predicate(bits)
    ) / distribution["total_weight"]


def wilson_interval(successes, total, z=1.959963984540054):
    successes = float(successes)
    total = float(total)
    if total <= 0 or successes < 0 or successes > total:
        return (np.nan, np.nan)
    proportion = successes / total
    denominator = 1.0 + z * z / total
    center = (
        proportion + z * z / (2.0 * total)
    ) / denominator
    half_width = (
        z
        * math.sqrt(
            proportion * (1.0 - proportion) / total
            + z * z / (4.0 * total * total)
        )
        / denominator
    )
    return (
        max(0.0, center - half_width),
        min(1.0, center + half_width),
    )


def returned_qasm_metrics(qasm_text):
    empty = {
        "returned_depth": np.nan,
        "returned_two_qubit_gates": np.nan,
        "returned_two_qubit_depth": np.nan,
        "returned_three_plus_qubit_gates": np.nan,
        "returned_swap_gates": np.nan,
        "returned_barriers": np.nan,
        "returned_num_qubits": np.nan,
        "returned_ops": {},
    }
    if not qasm_text:
        return {
            **empty,
            "returned_qasm_parse_reason":
                "missing_transpiled_qasm",
        }
    try:
        circuit = qasm2.loads(qasm_text)
    except Exception as exc:
        return {
            **empty,
            "returned_qasm_parse_reason":
                f"parse_error:{type(exc).__name__}:{exc}",
        }
    gates = [
        item for item in circuit.data
        if isinstance(item.operation, Gate)
    ]
    operations = {
        str(key): int(value)
        for key, value in circuit.count_ops().items()
    }
    return {
        "returned_depth": circuit.depth(),
        "returned_two_qubit_gates": sum(
            int(item.operation.num_qubits == 2)
            for item in gates
        ),
        "returned_two_qubit_depth": circuit.depth(
            filter_function=lambda item: (
                isinstance(item.operation, Gate)
                and item.operation.num_qubits == 2
            )
        ),
        "returned_three_plus_qubit_gates": sum(
            int(item.operation.num_qubits >= 3)
            for item in gates
        ),
        "returned_swap_gates": int(
            operations.get("swap", 0)
        ),
        "returned_barriers": sum(
            int(item.operation.name == "barrier")
            for item in circuit.data
        ),
        "returned_num_qubits": circuit.num_qubits,
        "returned_ops": operations,
        "returned_qasm_parse_reason": "ok",
    }


def parse_result_metrics(result, entry):
    family = entry["_family"]
    n_bits = int(
        entry.get(
            "n_total_qubits",
            entry.get("logical_qubits"),
        )
    )
    raw = normalize_distribution(
        result.get("count") or {},
        n_bits,
        require_integer=True,
    )
    corrected = normalize_distribution(
        result.get("corrected") or {},
        n_bits,
        require_integer=False,
    )
    shots_expected = int(entry["S_shots"])
    row = {
        "bitstring_order_assumption": BITSTRING_ORDER,
        "raw_counts_parse_reason": raw["parse_reason"],
        "corrected_parse_reason": corrected["parse_reason"],
        "raw_distribution": raw["weights"],
        "corrected_distribution": corrected["weights"],
        "dominant_returned_bitstring":
            raw["dominant_returned_bitstring"],
        "dominant_canonical_bitstring":
            raw["dominant_canonical_bitstring"],
        "shots_expected": shots_expected,
        "shots_observed": (
            int(raw["total_weight"])
            if raw["parse_reason"] == "ok"
            else 0
        ),
        "shots_match_manifest": (
            raw["parse_reason"] == "ok"
            and int(raw["total_weight"]) == shots_expected
        ),
        "raw_F_echo": np.nan,
        "raw_F_echo_ci_low": np.nan,
        "raw_F_echo_ci_high": np.nan,
        "corrected_F_echo": np.nan,
        "calibration_expected_bitstring": entry.get(
            "ideal_expected_bitstring"
        ),
        "calibration_expected_probability": np.nan,
        "calibration_ci_low": np.nan,
        "calibration_ci_high": np.nan,
        "calibration_dominant_matches_expected": None,
        "raw_success_probability": np.nan,
        "raw_success_probability_ci_low": np.nan,
        "raw_success_probability_ci_high": np.nan,
        "corrected_success_probability": np.nan,
        "raw_joint_probability": np.nan,
        "corrected_joint_probability": np.nan,
        "raw_conditional_fidelity": np.nan,
        "raw_conditional_fidelity_ci_low": np.nan,
        "raw_conditional_fidelity_ci_high": np.nan,
        "corrected_conditional_fidelity": np.nan,
        "raw_success_events": 0,
        "raw_F_echo_sem": np.nan,
        "raw_success_probability_sem": np.nan,
        "raw_conditional_fidelity_sem": np.nan,
        "directional_signal_ci_above_uniform_1_over_4": None,
    }
    if family == "bit_order_calibration":
        expected_bits = entry["ideal_expected_bitstring"]
        calibration_probability = probability(
            raw, lambda bits: bits == expected_bits
        )
        row["calibration_expected_probability"] = (
            calibration_probability
        )
        row["calibration_dominant_matches_expected"] = (
            raw["dominant_canonical_bitstring"]
            == expected_bits
            if raw["parse_reason"] == "ok"
            else None
        )
        if math.isfinite(calibration_probability):
            expected_events = (
                calibration_probability * raw["total_weight"]
            )
            low, high = wilson_interval(
                expected_events, raw["total_weight"]
            )
            row["calibration_ci_low"] = low
            row["calibration_ci_high"] = high
    elif family == "flat":
        raw_f = probability(raw, lambda bits: bits == "00")
        corrected_f = probability(
            corrected, lambda bits: bits == "00"
        )
        row["raw_F_echo"] = raw_f
        row["corrected_F_echo"] = corrected_f
        if math.isfinite(raw_f):
            row["raw_F_echo_sem"] = math.sqrt(
                raw_f * (1.0 - raw_f) / raw["total_weight"]
            )
            low, high = wilson_interval(
                raw_f * raw["total_weight"],
                raw["total_weight"],
            )
            row["raw_F_echo_ci_low"] = low
            row["raw_F_echo_ci_high"] = high
    else:
        raw_success = probability(
            raw, lambda bits: bits[:2] == "00"
        )
        corrected_success = probability(
            corrected, lambda bits: bits[:2] == "00"
        )
        raw_joint = probability(
            raw, lambda bits: bits == "0000"
        )
        corrected_joint = probability(
            corrected, lambda bits: bits == "0000"
        )
        raw_success_events = (
            sum(
                value
                for bits, value in raw["weights"].items()
                if bits[:2] == "00"
            )
            if raw["parse_reason"] == "ok"
            else 0.0
        )
        row["raw_success_probability"] = raw_success
        row["corrected_success_probability"] = (
            corrected_success
        )
        row["raw_success_events"] = int(raw_success_events)
        row["raw_joint_probability"] = raw_joint
        row["corrected_joint_probability"] = corrected_joint
        if math.isfinite(raw_success):
            row["raw_success_probability_sem"] = math.sqrt(
                raw_success
                * (1.0 - raw_success)
                / raw["total_weight"]
            )
            low, high = wilson_interval(
                raw_success * raw["total_weight"],
                raw["total_weight"],
            )
            row["raw_success_probability_ci_low"] = low
            row["raw_success_probability_ci_high"] = high
        is_verifier = entry.get("circuit_kind") == "verify"
        if is_verifier and raw_success_events > 0:
            raw_conditional = raw_joint / raw_success
            row["raw_conditional_fidelity"] = raw_conditional
            row["raw_conditional_fidelity_sem"] = math.sqrt(
                raw_conditional
                * (1.0 - raw_conditional)
                / raw_success_events
            )
            low, high = wilson_interval(
                raw_joint * raw["total_weight"],
                raw_success_events,
            )
            row["raw_conditional_fidelity_ci_low"] = low
            row["raw_conditional_fidelity_ci_high"] = high
            row[
                "directional_signal_ci_above_uniform_1_over_4"
            ] = bool(low > 0.25)
        if (
            is_verifier
            and math.isfinite(corrected_success)
            and corrected_success > 0
        ):
            row["corrected_conditional_fidelity"] = (
                corrected_joint / corrected_success
            )
    return {
        **row,
        **returned_qasm_metrics(result.get("transpiled", "")),
    }


# 不联网的 parser 自检。
assert normalize_distribution(
    {"00": 2, "11": 2}, 2, require_integer=True
)["parse_reason"] == "ok"
assert normalize_distribution(
    {"00": 1.5}, 2, require_integer=True
)["parse_reason"].startswith("noninteger_raw_count")
assert normalize_distribution(
    {"00": 0.8, "11": 0.2}, 2, require_integer=False
)["parse_reason"] == "ok"

calibration_mock_entry = entry_by_id[
    "cal_bitorder_2q_x_q0"
]
calibration_mock_returned = (
    "01" if BITSTRING_ORDER == "qiskit" else "10"
)
calibration_mock = parse_result_metrics(
    {
        "count": {
            calibration_mock_returned: 1000,
            "00": 24,
        },
        "corrected": {},
        "transpiled": (
            PROJECT_ROOT / calibration_mock_entry["qasm_path"]
        ).read_text(),
    },
    calibration_mock_entry,
)
assert calibration_mock[
    "calibration_dominant_matches_expected"
]
assert (
    calibration_mock["dominant_returned_bitstring"]
    == calibration_mock_returned
)

flat_mock_entry = entry_by_id[
    "imdb_flat_001_h_only_M0_r0"
]
flat_mock = parse_result_metrics(
    {
        "count": {"00": 900, "01": 40, "10": 40, "11": 44},
        "corrected": {},
        "transpiled": (
            PROJECT_ROOT / flat_mock_entry["qasm_path"]
        ).read_text(),
    },
    flat_mock_entry,
)
assert np.isclose(flat_mock["raw_F_echo"], 900 / 1024)
assert flat_mock["returned_qasm_parse_reason"] == "ok"

general_mock_entry = entry_by_id[
    "imdb_d4_general_qsvt_m8_verify"
]
general_mock_canonical_counts = {
    "0000": 30,
    "0001": 10,
    "0010": 10,
    "0011": 10,
    "1111": 964,
}
general_mock_returned_counts = {
    (
        bits if BITSTRING_ORDER == "qiskit" else bits[::-1]
    ): count
    for bits, count in general_mock_canonical_counts.items()
}
general_mock = parse_result_metrics(
    {
        "count": general_mock_returned_counts,
        "corrected": {},
        "transpiled": (
            PROJECT_ROOT / general_mock_entry["qasm_path"]
        ).read_text(),
    },
    general_mock_entry,
)
assert np.isclose(
    general_mock["raw_success_probability"], 60 / 1024
)
assert np.isclose(
    general_mock["raw_conditional_fidelity"], 0.5
)
assert 0 <= general_mock[
    "raw_conditional_fidelity_ci_low"
] <= general_mock[
    "raw_conditional_fidelity_ci_high"
] <= 1
print("Counts and returned-QASM parser self-tests PASS.")

Counts and returned-QASM parser self-tests PASS.


## 5. 建立可选网络会话并查询后端

只有 `ACTION` 为 `status`、`submit`、`submit_and_wait` 或 `fetch` 时才会创建
`Task`。`status()` 返回的非负整数是队列任务数；`Offline`、`Maintenance`、
缺失字段或未知结构全部 fail closed。

In [6]:
NETWORK_ACTIONS = {
    "status", "submit", "submit_and_wait", "fetch"
}
FINISHED_STATES = {"finished", "completed", "success"}
FAILED_STATES = {
    "failed", "error", "cancelled", "canceled", "deleted"
}


def normalized_status(value):
    return str(value).strip().lower()


def backend_value_is_available(value):
    queue_value = (
        isinstance(value, (int, np.integer))
        and not isinstance(value, (bool, np.bool_))
        and int(value) >= 0
    )
    online_label = (
        isinstance(value, str)
        and value.strip().lower()
        in {"online", "available", "idle"}
    )
    return queue_value or online_label


def valid_task_id(value):
    return (
        isinstance(value, (int, np.integer))
        and not isinstance(value, (bool, np.bool_))
        and int(value) > 0
    )


assert backend_value_is_available(0)
assert backend_value_is_available(57)
assert not backend_value_is_available(True)
assert not backend_value_is_available(-1)
assert not backend_value_is_available("Maintenance")
assert valid_task_id(123)
assert not valid_task_id(0)
assert not valid_task_id(True)
assert not valid_task_id("API error")

task_manager = None
live_backend_status = None
if ACTION in NETWORK_ACTIONS:
    from quafu import Task

    token = os.environ.get("QPU_API_TOKEN") or getpass(
        "请输入 Quafu-SQC token（输入不会显示）: "
    )
    task_manager = Task(token)
    if ACTION in {"status", "submit", "submit_and_wait"}:
        live_backend_status = task_manager.status()
        print("Live backend status:", live_backend_status)
        if not isinstance(live_backend_status, dict):
            raise RuntimeError(
                "Fail-closed: backend status is not a mapping."
            )
        backend_value = live_backend_status.get(BACKEND)
        if not backend_value_is_available(backend_value):
            raise RuntimeError(
                f"Fail-closed: unexpected {BACKEND} state "
                f"{backend_value!r}."
            )
        RAW_RESULT_DIR.mkdir(parents=True, exist_ok=True)
        status_path = (
            RAW_RESULT_DIR
            / (
                "backend_status_"
                + datetime.now(timezone.utc).strftime(
                    "%Y%m%dT%H%M%SZ"
                )
                + ".json"
            )
        )
        write_json_atomic(status_path, live_backend_status)
elif ACTION == "dry_run":
    print("Dry run: no token read, no network request.")

if ACTION == "status":
    print(
        f"{BACKEND} queue value:",
        live_backend_status.get(BACKEND),
    )

Dry run: no token read, no network request.


## 6. 提交

提交前会再次读取磁盘 QASM、重新计算 hash 和检查 schema。成功返回必须是正整数
task id；receipt 会立即落盘。`task_registry.json` 用于关闭 notebook 后继续取回
结果，不包含认证凭据。

In [7]:
def read_registry():
    if not REGISTRY_PATH.exists():
        return []
    payload = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
    if not isinstance(payload, list):
        raise ValueError("task_registry.json must contain a list.")
    return payload


def task_fingerprint(record):
    return json.dumps(
        {
            "run_label": record["run_label"],
            "experiment_id": record["experiment_id"],
            "backend": record["backend"],
            "compiler": record["compiler"],
            "correct": record["correct"],
            "open_dd": record["open_dd"],
            "target_qubits": record["target_qubits"],
        },
        sort_keys=True,
    )


def revalidate_selected_qasm(entry):
    qasm_path = PROJECT_ROOT / entry["qasm_path"]
    qasm_text = qasm_path.read_text(encoding="utf-8")
    observed_hash = hashlib.sha256(
        qasm_text.encode("utf-8")
    ).hexdigest()
    if observed_hash != entry["qasm_sha256"]:
        raise RuntimeError(
            f"QASM hash mismatch: {entry['experiment_id']}"
        )
    circuit = qasm2.loads(qasm_text)
    expected = int(
        entry.get(
            "n_total_qubits",
            entry.get("logical_qubits"),
        )
    )
    measurements = int(circuit.count_ops().get("measure", 0))
    if (
        circuit.num_qubits != expected
        or circuit.num_clbits != expected
        or measurements != expected
    ):
        raise RuntimeError(
            f"Schema mismatch: {entry['experiment_id']}"
        )
    return qasm_path, qasm_text, observed_hash


submitted_records = []
if ACTION in {"submit", "submit_and_wait"}:
    if CONFIRM_REAL_SUBMISSION != "RUN_QUAFU_HARDWARE":
        raise ValueError(
            "Set CONFIRM_REAL_SUBMISSION exactly to "
            "'RUN_QUAFU_HARDWARE'."
        )
    enabled_plan = [
        item for item in RUN_PLAN if bool(item["enabled"])
    ]
    if not enabled_plan:
        raise ValueError("No RUN_PLAN entry is enabled.")
    if len(enabled_plan) > int(MAX_TASKS_PER_SUBMISSION_CELL):
        raise ValueError(
            "Enabled task count exceeds "
            "MAX_TASKS_PER_SUBMISSION_CELL."
        )
    if any(
        entry_by_id[item["experiment_id"]]["_family"]
        == "general_qsvt"
        for item in enabled_plan
    ) and not ALLOW_DEEP_QSVT:
        raise ValueError(
            "A deep full-QSVT task is enabled. "
            "Set ALLOW_DEEP_QSVT=True only after controls pass."
        )
    if (
        any(
            entry_by_id[item["experiment_id"]]["_family"]
            == "general_qsvt"
            for item in enabled_plan
        )
        and REQUIRE_4Q_BIT_ORDER_CALIBRATION_FOR_QSVT
    ):
        completed_rows = []
        if HARDWARE_JSON_PATH.exists():
            payload = json.loads(
                HARDWARE_JSON_PATH.read_text(encoding="utf-8")
            )
            if isinstance(payload, list):
                completed_rows = payload
        required_calibrations = {
            "cal_bitorder_4q_x_q0",
            "cal_bitorder_4q_x_q3",
        }
        passed_calibrations = {
            item.get("experiment_id")
            for item in completed_rows
            if (
                item.get("experiment_id")
                in required_calibrations
                and item.get(
                    "calibration_dominant_matches_expected"
                )
                is True
                and item.get("bitstring_order_assumption")
                == BITSTRING_ORDER
                and normalized_status(
                    item.get("result_status")
                )
                in FINISHED_STATES
            )
        }
        missing_calibrations = (
            required_calibrations - passed_calibrations
        )
        if missing_calibrations:
            raise RuntimeError(
                "Full QSVT blocked until both 4q bit-order "
                "controls pass under the current "
                f"BITSTRING_ORDER. Missing: "
                f"{sorted(missing_calibrations)}"
            )

    registry = read_registry()
    existing_fingerprints = {
        task_fingerprint(item)
        for item in registry
        if valid_task_id(item.get("task_id"))
    }

    for plan_item in enabled_plan:
        experiment_id = plan_item["experiment_id"]
        entry = entry_by_id[experiment_id]
        compiler = plan_item.get(
            "compiler", DEFAULT_COMPILER
        )
        candidate = {
            "run_label": plan_item["run_label"],
            "experiment_id": experiment_id,
            "backend": BACKEND,
            "compiler": compiler,
            "correct": bool(USE_READOUT_CORRECTION),
            "open_dd": OPEN_DD,
            "target_qubits": [
                int(item) for item in TARGET_QUBITS
            ],
        }
        fingerprint = task_fingerprint(candidate)
        expected_qubits = int(
            entry.get(
                "n_total_qubits",
                entry.get("logical_qubits"),
            )
        )
        if (
            TARGET_QUBITS
            and len(TARGET_QUBITS) != expected_qubits
        ):
            raise ValueError(
                f"{plan_item['run_label']}: TARGET_QUBITS has "
                f"{len(TARGET_QUBITS)} entries, expected "
                f"{expected_qubits}."
            )
        if (
            fingerprint in existing_fingerprints
            and not ALLOW_DUPLICATE_SUBMISSION
        ):
            raise RuntimeError(
                "Duplicate submission blocked for "
                f"{plan_item['run_label']!r}. "
                "Change the label or explicitly allow duplicates."
            )

        qasm_path, qasm_text, observed_hash = (
            revalidate_selected_qasm(entry)
        )
        timestamp = datetime.now(timezone.utc)
        task_name = (
            f"qos_{plan_item['run_label']}_"
            f"{timestamp.strftime('%m%dT%H%M%S')}"
        )[:80]
        task = {
            "chip": BACKEND,
            "name": task_name,
            "circuit": qasm_text,
            "shots": int(entry["S_shots"]),
            "compile": True,
            "options": {
                "compiler": compiler,
                "correct": bool(USE_READOUT_CORRECTION),
                "open_dd": OPEN_DD,
                "target_qubits": [
                    int(item) for item in TARGET_QUBITS
                ],
            },
        }
        task_id = task_manager.run(task)
        if not valid_task_id(task_id):
            raise RuntimeError(
                "Submission did not return a positive integer "
                f"task id; received {task_id!r}."
            )
        task_id = int(task_id)
        record = {
            **candidate,
            "task_id": task_id,
            "task_name": task_name,
            "shots": int(entry["S_shots"]),
            "qasm_path": str(qasm_path.relative_to(PROJECT_ROOT)),
            "qasm_sha256": observed_hash,
            "submitted_at_utc": timestamp.isoformat(),
            "submission_status": "submitted",
            "result_status": None,
            "environment": ENVIRONMENT,
        }
        RAW_RESULT_DIR.mkdir(parents=True, exist_ok=True)
        receipt_path = (
            RAW_RESULT_DIR
            / f"receipt_{plan_item['run_label']}_{task_id}.json"
        )
        write_json_atomic(receipt_path, record)
        record["receipt_path"] = str(
            path_for_record(receipt_path)
        )
        registry.append(record)
        write_json_atomic(REGISTRY_PATH, registry)
        existing_fingerprints.add(fingerprint)
        submitted_records.append(record)
        print(
            "Submitted:",
            plan_item["run_label"],
            "task_id=",
            task_id,
        )
else:
    print("Submission cell inactive for ACTION =", ACTION)

Submission cell inactive for ACTION = dry_run


## 7. 可选轮询

`submit_and_wait` 会轮询 `status(task_id)`，最长等待由配置决定。`submit`
只保存 task id 后立即结束，更适合队列较长时关闭 notebook、稍后再用
`ACTION="fetch"` 取回。

In [8]:
latest_status_by_task = {}

if ACTION == "submit_and_wait" and submitted_records:
    pending = {
        int(record["task_id"]): record
        for record in submitted_records
    }
    start = time.monotonic()
    while pending:
        for task_id in list(pending):
            status_value = task_manager.status(task_id)
            latest_status_by_task[task_id] = status_value
            state = normalized_status(status_value)
            print(task_id, status_value)
            if state in FINISHED_STATES | FAILED_STATES:
                pending.pop(task_id)
        if not pending:
            break
        elapsed = time.monotonic() - start
        if elapsed >= float(MAX_WAIT_SECONDS):
            print("Polling timeout; unfinished:", sorted(pending))
            break
        time.sleep(float(POLL_INTERVAL_SECONDS))
else:
    print("Polling cell inactive.")

Polling cell inactive.


## 8. 取回 raw result、returned circuit 与指标

重新打开 notebook 时，把 `ACTION="fetch"`，并在顶部填写
`FETCH_RUN_LABELS`。如果 task 并非由这本 notebook 提交，则填写
`MANUAL_FETCH_PLAN`。

未完成任务默认只记录状态，不强行调用 result endpoint。

In [9]:
def latest_registry_records_for_labels(labels):
    registry = read_registry()
    if labels == ["*"]:
        selected = registry
    else:
        selected = []
        for label in labels:
            matches = [
                item
                for item in registry
                if item.get("run_label") == label
            ]
            if not matches:
                raise ValueError(
                    f"No registry task for run_label {label!r}."
                )
            selected.append(matches[-1])
    return selected


def validate_fetch_record(record):
    required = {
        "run_label", "experiment_id", "compiler", "task_id"
    }
    missing = required - set(record)
    if missing:
        raise ValueError(
            f"Fetch record missing fields: {sorted(missing)}"
        )
    if record["experiment_id"] not in entry_by_id:
        raise ValueError(
            f"Unknown experiment id: {record['experiment_id']}"
        )
    if not valid_task_id(record["task_id"]):
        raise ValueError(
            f"Invalid task id: {record['task_id']!r}"
        )
    if not re.fullmatch(
        r"[A-Za-z0-9_-]+", str(record["run_label"])
    ):
        raise ValueError(
            f"Unsafe run_label: {record['run_label']!r}"
        )
    if record["compiler"] not in ALLOWED_COMPILERS:
        raise ValueError(
            f"Unsupported compiler: {record['compiler']!r}"
        )
    return {
        **record,
        "task_id": int(record["task_id"]),
        "backend": record.get("backend", BACKEND),
    }


fetch_records = []
if ACTION == "submit_and_wait":
    fetch_records.extend(submitted_records)
elif ACTION == "fetch":
    if FETCH_RUN_LABELS:
        fetch_records.extend(
            latest_registry_records_for_labels(FETCH_RUN_LABELS)
        )
    fetch_records.extend(MANUAL_FETCH_PLAN)
    if not fetch_records:
        raise ValueError(
            "Set FETCH_RUN_LABELS or MANUAL_FETCH_PLAN first."
        )

deduplicated_fetch_records = []
seen_task_ids = set()
for raw_record in fetch_records:
    record = validate_fetch_record(raw_record)
    if record["task_id"] not in seen_task_ids:
        seen_task_ids.add(record["task_id"])
        deduplicated_fetch_records.append(record)
fetch_records = deduplicated_fetch_records

new_result_rows = []
if ACTION in {"fetch", "submit_and_wait"}:
    RAW_RESULT_DIR.mkdir(parents=True, exist_ok=True)
    for record in fetch_records:
        task_id = int(record["task_id"])
        experiment_id = record["experiment_id"]
        entry = entry_by_id[experiment_id]

        status_value = latest_status_by_task.get(task_id)
        if status_value is None:
            status_value = task_manager.status(task_id)
        state = normalized_status(status_value)
        base_row = {
            "fetched_at_utc":
                datetime.now(timezone.utc).isoformat(),
            "run_label": record["run_label"],
            "experiment_id": experiment_id,
            "family": entry["_family"],
            "control_type": entry.get(
                "control_type", entry.get("circuit_kind")
            ),
            "task_id": task_id,
            "backend": record.get("backend", BACKEND),
            "compiler": record["compiler"],
            "task_status_query": status_value,
            "ideal_expected_bitstring": entry.get(
                "ideal_expected_bitstring"
            ),
            "ideal_expected_probability": entry.get(
                "ideal_expected_probability"
            ),
            "ideal_F_echo": entry.get("ideal_F_echo"),
            "ideal_success_probability": entry.get(
                "ideal_success_probability"
            ),
            "ideal_conditional_fidelity": entry.get(
                "ideal_conditional_fidelity"
            ),
        }
        if (
            state not in FINISHED_STATES
            and not FORCE_FETCH_NONFINISHED
        ):
            print(
                "Not finished; skipped result endpoint:",
                task_id,
                status_value,
            )
            new_result_rows.append(
                {
                    **base_row,
                    "result_status": None,
                    "result_error": "",
                    "fetch_reason":
                        f"nonfinished_status:{status_value!r}",
                }
            )
            continue

        result = task_manager.result(
            task_id, timeout=float(RESULT_TIMEOUT_SECONDS)
        )
        if not isinstance(result, dict):
            result = {
                "status": "invalid_result_type",
                "finished": False,
                "error":
                    f"Expected mapping, got "
                    f"{type(result).__name__}: {result!r}",
            }
        result_payload_terminal = (
            normalized_status(result.get("status"))
            in FINISHED_STATES
        )
        result_payload_has_counts = (
            isinstance(result.get("count"), dict)
            and bool(result.get("count"))
        )
        result_payload_reason = (
            "ok"
            if (
                result_payload_terminal
                and result_payload_has_counts
            )
            else (
                "nonterminal_or_empty_result_payload:"
                f"status={result.get('status')!r},"
                f"has_counts={result_payload_has_counts}"
            )
        )
        raw_path = (
            RAW_RESULT_DIR
            / f"result_{record['run_label']}_{task_id}.json"
        )
        write_json_atomic(raw_path, result)

        returned_qasm_value = result.get("transpiled", "")
        if returned_qasm_value is None or returned_qasm_value == "":
            returned_qasm = ""
            returned_qasm_field_reason = "missing"
        elif isinstance(returned_qasm_value, str):
            returned_qasm = returned_qasm_value
            returned_qasm_field_reason = "ok"
        else:
            returned_qasm = ""
            returned_qasm_field_reason = (
                "invalid_type:"
                f"{type(returned_qasm_value).__name__}"
            )
        returned_qasm_path = None
        returned_qasm_sha256 = None
        if returned_qasm:
            returned_qasm_path = (
                RAW_RESULT_DIR
                / (
                    f"returned_{record['run_label']}_"
                    f"{task_id}.qasm"
                )
            )
            returned_qasm_path.write_text(
                returned_qasm, encoding="utf-8"
            )
            returned_qasm_sha256 = hashlib.sha256(
                returned_qasm.encode("utf-8")
            ).hexdigest()

        returned_tid = result.get("tid")
        task_id_matches = (
            returned_tid is None
            or (
                valid_task_id(returned_tid)
                and int(returned_tid) == task_id
            )
        )
        metrics = parse_result_metrics(result, entry)
        new_result_rows.append(
            {
                **base_row,
                "result_status": result.get("status"),
                "result_payload_terminal":
                    result_payload_terminal,
                "result_payload_has_counts":
                    result_payload_has_counts,
                "result_finished": result.get("finished"),
                "result_error": result.get("error", ""),
                "returned_task_id": returned_tid,
                "task_id_matches": task_id_matches,
                "raw_result_path": str(
                    path_for_record(raw_path)
                ),
                "returned_qasm_path": (
                    path_for_record(returned_qasm_path)
                    if returned_qasm_path else None
                ),
                "returned_qasm_sha256": returned_qasm_sha256,
                "returned_qasm_field_reason":
                    returned_qasm_field_reason,
                "fetch_reason": result_payload_reason,
                **metrics,
            }
        )

    existing_rows = []
    if HARDWARE_JSON_PATH.exists():
        payload = json.loads(
            HARDWARE_JSON_PATH.read_text(encoding="utf-8")
        )
        if isinstance(payload, list):
            existing_rows = payload
    combined = {
        int(item["task_id"]): item
        for item in existing_rows
        if valid_task_id(item.get("task_id"))
    }
    for item in new_result_rows:
        combined[int(item["task_id"])] = item
    combined_rows = list(combined.values())
    write_json_atomic(HARDWARE_JSON_PATH, combined_rows)
    hardware_df = pd.DataFrame(combined_rows)
    hardware_df.to_csv(HARDWARE_CSV_PATH, index=False)
    print("Saved result rows:", len(combined_rows))
else:
    print("Fetch cell inactive.")

Fetch cell inactive.


## 9. 汇总与可视化

这张表必须和 raw JSON、returned QASM 一起解释。特别是 full QSVT：

- 当前 `M=8` 可提交 verifier 的理想值约为
  $p_{\rm succ}=0.05124$、$F_{\rm cond}=0.46328$；
- `10` 中 $F_{\rm cond}=0.9961$ 是另一条固定 balanced 回归流，不是这条
  `M=8` IID QASM 的预期值；
- 条件 fidelity 必须同时报告 success events，否则可能只来自很少的后选择样本。

In [10]:
if HARDWARE_JSON_PATH.exists():
    hardware_rows = json.loads(
        HARDWARE_JSON_PATH.read_text(encoding="utf-8")
    )
    hardware_df = pd.DataFrame(hardware_rows)
else:
    hardware_df = pd.DataFrame()

if hardware_df.empty:
    print(
        "No hardware result yet. This is expected in dry_run/status mode."
    )
else:
    important_columns = [
        "run_label",
        "experiment_id",
        "task_id",
        "compiler",
        "result_status",
        "fetch_reason",
        "bitstring_order_assumption",
        "calibration_expected_bitstring",
        "dominant_returned_bitstring",
        "dominant_canonical_bitstring",
        "calibration_expected_probability",
        "calibration_ci_low",
        "calibration_ci_high",
        "calibration_dominant_matches_expected",
        "raw_distribution",
        "ideal_F_echo",
        "raw_F_echo",
        "raw_F_echo_ci_low",
        "raw_F_echo_ci_high",
        "ideal_success_probability",
        "raw_success_probability",
        "raw_success_probability_ci_low",
        "raw_success_probability_ci_high",
        "raw_success_events",
        "raw_joint_probability",
        "ideal_conditional_fidelity",
        "raw_conditional_fidelity",
        "raw_conditional_fidelity_ci_low",
        "raw_conditional_fidelity_ci_high",
        "directional_signal_ci_above_uniform_1_over_4",
        "shots_match_manifest",
        "returned_two_qubit_gates",
        "returned_two_qubit_depth",
        "returned_swap_gates",
        "returned_qasm_field_reason",
        "returned_qasm_parse_reason",
    ]
    available_columns = [
        column
        for column in important_columns
        if column in hardware_df.columns
    ]
    display(hardware_df[available_columns])

    metric_specs = [
        (
            "raw_F_echo",
            "ideal_F_echo",
            "Flat echo fidelity",
        ),
        (
            "raw_success_probability",
            "ideal_success_probability",
            "General QSVT heralding",
        ),
        (
            "raw_conditional_fidelity",
            "ideal_conditional_fidelity",
            "General QSVT conditional fidelity",
        ),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    plotted = False
    for axis, (measured, ideal, title) in zip(
        axes, metric_specs
    ):
        if measured not in hardware_df:
            axis.set_axis_off()
            continue
        subset = hardware_df.loc[
            pd.to_numeric(
                hardware_df[measured], errors="coerce"
            ).notna()
        ].copy()
        if subset.empty:
            axis.set_axis_off()
            continue
        plotted = True
        x = np.arange(len(subset))
        observed = pd.to_numeric(
            subset[measured], errors="coerce"
        ).to_numpy()
        expected = pd.to_numeric(
            subset[ideal], errors="coerce"
        ).to_numpy()
        axis.scatter(x, observed, s=70, label="hardware raw")
        axis.scatter(
            x,
            expected,
            s=70,
            marker="x",
            label="ideal manifest",
        )
        axis.set_xticks(x)
        axis.set_xticklabels(
            subset["run_label"], rotation=35, ha="right"
        )
        axis.set_ylim(-0.02, 1.02)
        axis.set_title(title)
        axis.grid(alpha=0.25)
        axis.legend()
    if plotted:
        fig.tight_layout()
        plot_path = (
            RAW_RESULT_DIR / "hardware_validation_results.png"
        )
        fig.savefig(plot_path, dpi=180, bbox_inches="tight")
        print("Saved plot:", plot_path)
        plt.show()
    else:
        plt.close(fig)
        print("No completed metric rows to plot yet.")

No hardware result yet. This is expected in dry_run/status mode.


## 10. 真机结论的最低报告要求

每条任务至少保存并报告：

1. backend、compiler、shots、task id 和 finished timestamp；
2. 输入 QASM hash；
3. raw counts；若使用 correction，再单列 corrected distribution；
4. returned transpiled QASM/hash；
5. 实际二比特门数、二比特 depth、SWAP 和三比特门；
6. flat 的 $F_{\rm echo}$，或 full QSVT 的
   $p_{\rm succ}$、success events、$F_{\rm cond}$；
7. 95% Wilson 区间、对应 manifest 理想值与误差定义；
8. one-hot calibration 确认的 bitstring order。

真机小规模结果可以证明门级机制在设备上的可执行性和噪声代价，但不能单独证明
IMDb 规模的端到端指数空间优势。先按
`2q/4q bit-order → H-only → exact/balanced → compiler A/B → one IID
→ M8 prepare/verify`
推进，完整操作见项目根目录的 `HARDWARE_README.md`。